# Reusable Immunofluorescence Quantification and Control-Referenced Z-score Normalization

**Purpose.** Process CellProfiler measurements from different antibody panels and independent experiments using a single workflow. Select the CellProfiler output folder or individual CSV files, associate measurements with treatment metadata, calculate well-level mean measurements, and normalize each feature relative to the untreated controls.

**Source methods.** This notebook is adapted from the original LC3/MTMR4 analysis. It preserves object-level-to-well-level mean aggregation and the Excel normalization formula `(value − control mean) / control population standard deviation` (`STDEV.P`, equivalent to `ddof=0`).

**Experimental unit.** Each exported row represents one well. The workflow does **not** pool independent experiments for normalization: run it separately for each experiment to retain its own control reference.

## 1. Imports and experiment configuration

Choose an antibody-panel label and experiment identifier for output traceability. By default, a file dialog opens to select the CellProfiler results folder; alternatively, specify the folder path directly. Individual file-selection dialogs are also available when the files are stored in different folders.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ANTIBODY_PANEL = "LC3_MTMR4"  # Change for each antibody panel
EXPERIMENT_ID = "exp2"        # Change for each independent experiment
CONTROL_LABEL = "Non_Treatment"
CONTROL_DDOF = 0             # Excel STDEV.P
EXPECTED_CONTROL_WELLS = 3   # Reference workbook uses three control wells

# Option A: provide a directory path; Option B: leave None to choose interactively.
DATA_DIR = None
# DATA_DIR = Path(r"C:/path/to/experiment/Resultados")
OUTPUT_ROOT = Path("immunofluorescence_results")

# For other antibody panels, the available features are detected automatically.
# Add exact column names here to restrict the exported features, or leave None for all
# intensity, area, and puncta-count measurements present in the CellProfiler exports.
FEATURES_BY_COMPARTMENT = {
    "Nuclei": None,
    "Cells": None,
    "Cytoplasm": None,
}
TREATMENT_ORDER = [
    "Non_Treatment", "AuNP", "AuNP_+_Healthy_Serum",
    "AuNP_+_Breast_Cancer_and_Cachexia_Serum",
    "Healthy_Serum", "Breast_Cancer_and_Cachexia_Serum"
]

## 2. Select the CellProfiler exports and experimental plate map

The original LC3/MTMR4 analysis reads `Image.csv`, `Nuclei.csv`, `Cells.csv`, `Cytoplasm.csv`, and a separate plate-map CSV. This reusable version finds the four standard CellProfiler files automatically inside the selected folder, and asks for the plate map when needed.

The image-to-well mapping uses the `Well` field when present; otherwise it extracts identifiers such as `B1` or `B01` from `FileName_GFP` (the source notebook's method). It checks for unmatched images and duplicate plate-map wells.

In [ ]:
def select_directory(prompt):
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askdirectory(title=prompt)
        root.destroy()
    except Exception as exc:
        raise RuntimeError(
            "Folder dialog is unavailable. Set DATA_DIR to the folder path above."
        ) from exc
    if not selected:
        raise ValueError("No folder was selected.")
    return Path(selected)

def select_csv(prompt):
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title=prompt, filetypes=[("CSV files", "*.csv")]
        )
        root.destroy()
    except Exception as exc:
        raise RuntimeError(
            "File dialog is unavailable. Specify the CSV path directly."
        ) from exc
    if not selected:
        raise ValueError(f"No file selected: {prompt}")
    return Path(selected)

def find_export(directory, filename):
    matches = sorted(
        p for p in directory.rglob("*.csv") if p.name.lower() == filename.lower()
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename} under {directory}, found {len(matches)}. "
            "Use the manual file-selection option below."
        )
    return matches[0]

data_dir = Path(DATA_DIR) if DATA_DIR is not None else select_directory(
    "Select the CellProfiler experiment results folder"
)
data_dir = data_dir.expanduser().resolve()
print("Selected experiment folder:", data_dir)

# Set USE_INDIVIDUAL_FILE_DIALOGS=True if your files are in different folders.
USE_INDIVIDUAL_FILE_DIALOGS = False
if USE_INDIVIDUAL_FILE_DIALOGS:
    paths = {
        name: select_csv(f"Select {name}.csv")
        for name in ("Image", "Nuclei", "Cells", "Cytoplasm")
    }
else:
    paths = {
        name: find_export(data_dir, f"{name}.csv")
        for name in ("Image", "Nuclei", "Cells", "Cytoplasm")
    }

plate_candidates = [
    p for p in data_dir.rglob("*.csv")
    if "platmap" in p.name.lower() or "platemap" in p.name.lower()
]
PLATEMAP_PATH = None  # Set a CSV path here to skip the plate-map dialog.
if PLATEMAP_PATH is not None:
    paths["Platemap"] = Path(PLATEMAP_PATH)
elif len(plate_candidates) == 1:
    paths["Platemap"] = plate_candidates[0]
else:
    paths["Platemap"] = select_csv("Select the experimental plate-map CSV")

for name, path in paths.items():
    print(f"{name:10s}: {path}")

## 3. Validate metadata and match images to treatment groups

Well identifiers are standardized to the three-character form (`B1` → `B01`). The plate map must contain `Well` and `Treatment` columns. Each image must match a plate-map well and each `ImageNumber` must be unique within the selected experiment.

In [ ]:
def standardize_well(value):
    if pd.isna(value):
        return pd.NA
    match = re.fullmatch(r"([A-Za-z])(\d{1,2})", str(value).strip())
    return f"{match.group(1).upper()}{int(match.group(2)):02d}" if match else pd.NA

image_df = pd.read_csv(paths["Image"], low_memory=False)
platemap = pd.read_csv(paths["Platemap"], low_memory=False)
for col in ("Well", "Treatment"):
    if col not in platemap.columns:
        raise KeyError(f"Missing plate-map column: {col}")
if "ImageNumber" not in image_df.columns:
    raise KeyError("Image.csv is missing ImageNumber.")

platemap["Well"] = platemap["Well"].map(standardize_well)
if platemap["Well"].isna().any():
    raise ValueError("Invalid or missing well identifiers in the plate map.")
if platemap["Well"].duplicated().any():
    raise ValueError("Plate map contains duplicate wells; resolve before merging.")

if "Well" in image_df.columns:
    image_df["Well"] = image_df["Well"].map(standardize_well)
elif "FileName_GFP" in image_df.columns:
    extracted = image_df["FileName_GFP"].astype(str).str.extract(
        r"(?<![A-Za-z0-9])([A-Za-z]\d{1,2})(?!\d)", expand=False
    )
    image_df["Well"] = extracted.map(standardize_well)
else:
    raise KeyError("Image.csv requires Well or FileName_GFP for well identification.")

if image_df["Well"].isna().any():
    raise ValueError("Some image wells could not be identified.")
if image_df["ImageNumber"].duplicated().any():
    raise ValueError("Duplicate ImageNumber values in Image.csv.")

image_metadata = image_df.merge(
    platemap[["Well", "Treatment"]], on="Well",
    how="left", validate="many_to_one", indicator=True
)
if (image_metadata["_merge"] != "both").any():
    missing = image_metadata.loc[
        image_metadata["_merge"] != "both", "Well"
    ].unique().tolist()
    raise ValueError(f"Image wells missing from plate map: {missing}")
image_metadata = image_metadata.drop(columns="_merge")
print("Images:", len(image_metadata))
print("Wells:", image_metadata["Well"].nunique())
print(image_metadata.groupby("Treatment", observed=True)["Well"].nunique())

## 4. Aggregate measurements separately for nuclei, cells, and cytoplasm

Object-level CellProfiler measurements are linked to the corresponding image by `ImageNumber`, then averaged across all objects in each well and treatment group. This reproduces the source notebook's `groupby(['Well', 'Treatment']).mean()` logic.

The selected feature set is configurable for different antibody panels. By default, the workflow retains intensity measurements, area measurements, and `Children_*_Count` puncta measurements, without assuming that all antibodies share the LC3/MTMR4 channel names. Missing requested features cause an explicit error rather than being silently skipped.

In [ ]:
def available_features(columns):
    return [
        c for c in columns
        if (
            c.startswith("Intensity_")
            or c.startswith("AreaShape_")
            or (c.startswith("Children_") and c.endswith("_Count"))
        )
    ]

def aggregate_compartment(compartment, file_path, metadata, requested=None):
    objects = pd.read_csv(file_path, low_memory=False)
    if "ImageNumber" not in objects.columns:
        raise KeyError(f"{compartment}.csv is missing ImageNumber.")
    features = available_features(objects.columns) if requested is None else list(requested)
    missing = [c for c in features if c not in objects.columns]
    if missing:
        raise KeyError(f"{compartment}: requested features not found: {missing}")
    if not features:
        raise ValueError(f"{compartment}: no intensity, area, or puncta features found.")

    # Retain object-level columns only; avoids the ambiguous '_x'/'_y' suffixes
    # created when the entire Image.csv is merged with object measurements.
    object_values = objects[["ImageNumber"] + features].copy()
    object_values[features] = object_values[features].apply(
        pd.to_numeric, errors="coerce"
    )
    image_labels = metadata[["ImageNumber", "Well", "Treatment"]]
    merged = object_values.merge(
        image_labels, on="ImageNumber", how="left",
        validate="many_to_one", indicator=True
    )
    if (merged["_merge"] != "both").any():
        raise ValueError(f"{compartment}: some object ImageNumber values have no image.")
    merged = merged.drop(columns="_merge")

    summary = merged.groupby(
        ["Well", "Treatment"], as_index=False, observed=True
    )[features].mean()
    object_counts = merged.groupby(
        ["Well", "Treatment"], observed=True
    ).size().rename("Object_Count").reset_index()
    summary = summary.merge(
        object_counts, on=["Well", "Treatment"], validate="one_to_one"
    )
    return summary, features

compartment_tables = {}
feature_sets = {}
for compartment in ("Nuclei", "Cells", "Cytoplasm"):
    summary, features = aggregate_compartment(
        compartment, paths[compartment], image_metadata,
        requested=FEATURES_BY_COMPARTMENT[compartment]
    )
    compartment_tables[compartment] = summary
    feature_sets[compartment] = features
    print(f"{compartment}: {len(summary)} wells, {len(features)} features")

count_features = [
    c for c in ("Count_Nuclei", "Count_Cells")
    if c in image_metadata.columns
]
image_counts = (
    image_metadata.groupby(["Well", "Treatment"], as_index=False)[count_features].mean()
    if count_features else pd.DataFrame()
)

## 5. Normalize well-level measurements against the negative control

For each compartment and feature, the control mean and **population** standard deviation are computed using the `Non_Treatment` wells of the selected experiment. Every well is transformed with the same control-derived parameters.

This reproduces the intended Excel formula `=(C3-AVERAGE($C$3:$C$5))/STDEV.P($C$3:$C$5)` when the three referenced rows are the untreated control wells. Features with zero or undefined control standard deviation are reported as missing rather than divided by zero.

**Important:** Three untreated wells are expected from the example spreadsheet. The notebook warns if their number differs, rather than quietly pooling other experiments.

In [ ]:
def control_zscore(table, features, control_label=CONTROL_LABEL):
    control = table.loc[table["Treatment"].astype(str) == control_label]
    if control.empty:
        raise ValueError(f"No '{control_label}' control wells found.")
    if len(control) != EXPECTED_CONTROL_WELLS:
        print(
            f"WARNING: found {len(control)} control wells; "
            f"the Excel reference contains {EXPECTED_CONTROL_WELLS}."
        )

    values = table[features].apply(pd.to_numeric, errors="coerce")
    ref = control[features].apply(pd.to_numeric, errors="coerce")
    means = ref.mean(axis=0)
    stds = ref.std(axis=0, ddof=CONTROL_DDOF)
    valid = stds.notna() & (stds != 0)
    if (~valid).any():
        print("WARNING: undefined control standard deviation for:",
              stds.index[~valid].tolist())

    zvalues = values.sub(means, axis="columns").div(stds.where(valid), axis="columns")
    normalized = table[["Well", "Treatment"]].copy()
    normalized[features] = zvalues
    params = pd.DataFrame({
        "Feature": features,
        "Control_Mean": means.reindex(features).to_numpy(),
        "Control_SD_Population": stds.reindex(features).to_numpy(),
        "Control_N": ref.notna().sum().reindex(features).to_numpy(),
    })
    return normalized, params

zscore_tables, normalization_parameters = {}, {}
for compartment, table in compartment_tables.items():
    zscore_tables[compartment], normalization_parameters[compartment] = (
        control_zscore(table, feature_sets[compartment])
    )
    print(f"{compartment}: Z-scores calculated for {len(feature_sets[compartment])} features.")

## 6. Export Prism-ready tables and normalization parameters

All outputs are saved under a folder named for the antibody panel and independent experiment. The Excel workbook includes separate sheets for raw well-level means, normalized Z-scores, and control parameters for each cellular compartment. CSV versions are also exported.

The outputs are intended for downstream analysis in GraphPad Prism. This notebook does not invent or automatically apply a statistical test to the experimental groups.

In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")

output_dir = OUTPUT_ROOT / safe_name(ANTIBODY_PANEL) / safe_name(EXPERIMENT_ID)
output_dir.mkdir(parents=True, exist_ok=True)
output_xlsx = output_dir / "immunofluorescence_well_means_and_zscores.xlsx"

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    for compartment in ("Nuclei", "Cells", "Cytoplasm"):
        compartment_tables[compartment].to_excel(
            writer, sheet_name=f"{compartment}_Means", index=False
        )
        zscore_tables[compartment].to_excel(
            writer, sheet_name=f"{compartment}_Zscore", index=False
        )
        normalization_parameters[compartment].to_excel(
            writer, sheet_name=f"{compartment}_Control", index=False
        )
        compartment_tables[compartment].to_csv(
            output_dir / f"{compartment.lower()}_well_means.csv", index=False
        )
        zscore_tables[compartment].to_csv(
            output_dir / f"{compartment.lower()}_zscores.csv", index=False
        )
    if not image_counts.empty:
        image_counts.to_excel(writer, sheet_name="Image_Counts", index=False)

print("Saved workbook:", output_xlsx.resolve())
print("Results folder:", output_dir.resolve())

## 7. Colored boxplots, one-way ANOVA, Tukey HSD, and statistical letters

Select a compartment and feature. The graph preserves the experiment-specific, control-referenced Z-scores, uses a consistent treatment palette, overlays individual **well means**, and displays compact-letter annotations from Tukey HSD (α = 0.05). Groups sharing at least one letter were not significantly different in Tukey's test; this does **not** demonstrate equivalence.

**Experimental unit:** each dot represents one well, not an individual cell. ANOVA/Tukey here treats wells as independent units **within this experiment**. If wells are technical replicates or if several independent experiments are combined, aggregate appropriately or use a model accounting for experiment/plate effects before making biological inferences. Check the assumptions of one-way ANOVA. Results and figures are exported to the existing experiment output folder.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations

PLOT_COMPARTMENT = "Cells"
PLOT_FEATURE = "Intensity_IntegratedIntensity_CY5"  # Set to None for the first available feature.
ALPHA = 0.05

# Original ten-color palette; colors are mapped to TREATMENT_ORDER.
CUSTOM_COLORS = [
    "#272323", "#1730F0", "#0a17a4", "#08024C", "#f79d7c",
    "#ef5946", "#FB0505", "#784f48", "#c93d28", "#651105",
]
COLOR_MAP = {
    treatment: CUSTOM_COLORS[i % len(CUSTOM_COLORS)]
    for i, treatment in enumerate(TREATMENT_ORDER)
}


def compact_letter_display(group_names, significant_pairs):
    """Assign letters: significant pairs share none; non-significant pairs share >=1."""
    columns = [set(group_names)]
    for a, b in combinations(group_names, 2):
        if frozenset((a, b)) not in significant_pairs:
            continue
        split = []
        for column in columns:
            if a in column and b in column:
                split.extend((column - {a}, column - {b}))
            else:
                split.append(column)
        unique = []
        for column in split:
            if column and column not in unique:
                unique.append(column)
        columns = [
            column for column in unique
            if not any(column < other for other in unique)
        ]
    columns.sort(key=lambda col: min(group_names.index(g) for g in col))

    def letter_at(index):
        label = ""
        index += 1
        while index:
            index, remainder = divmod(index - 1, 26)
            label = chr(97 + remainder) + label
        return label

    result = {name: "" for name in group_names}
    for i, column in enumerate(columns):
        for group in column:
            result[group] += letter_at(i)
    return result


plot_table = zscore_tables[PLOT_COMPARTMENT].copy()
plot_feature = PLOT_FEATURE or feature_sets[PLOT_COMPARTMENT][0]
if plot_feature not in plot_table.columns:
    raise KeyError(f"Feature not found: {plot_feature}")

plot_table["Treatment"] = plot_table["Treatment"].astype(str)
plot_table[plot_feature] = pd.to_numeric(plot_table[plot_feature], errors="coerce")
plot_table = plot_table.dropna(subset=["Treatment", plot_feature])

present = plot_table["Treatment"].unique().tolist()
ordered = [t for t in TREATMENT_ORDER if t in present]
ordered += [t for t in present if t not in ordered]
values = [
    plot_table.loc[plot_table["Treatment"] == t, plot_feature].to_numpy()
    for t in ordered
]
if not values:
    raise ValueError(f"No valid observations for {plot_feature}.")

anova_f, anova_p = np.nan, np.nan
tukey_table = pd.DataFrame()
letters = {t: "" for t in ordered}

if len(ordered) >= 2 and all(len(v) >= 2 for v in values):
    anova = stats.f_oneway(*values)
    anova_f, anova_p = float(anova.statistic), float(anova.pvalue)
    if np.isfinite(anova_p):
        tukey = pairwise_tukeyhsd(
            endog=plot_table[plot_feature],
            groups=plot_table["Treatment"],
            alpha=ALPHA,
        )
        tukey_table = pd.DataFrame(
            tukey.summary().data[1:], columns=tukey.summary().data[0]
        )
        significant_pairs = {
            frozenset((str(row["group1"]), str(row["group2"])))
            for _, row in tukey_table.iterrows()
            if str(row["reject"]).lower() == "true"
        }
        letters = compact_letter_display(ordered, significant_pairs)
        print(f"One-way ANOVA: F={anova_f:.4g}, p={anova_p:.4g}")
        print(tukey_table.to_string(index=False))
    else:
        print("ANOVA returned a non-finite p-value; Tukey and letters omitted.")
else:
    print("ANOVA/Tukey omitted: at least two groups with >=2 wells each are required.")

fig, ax = plt.subplots(figsize=(max(9, len(ordered) * 1.7), 6))
boxes = ax.boxplot(
    values, labels=ordered, patch_artist=True, showfliers=False,
    widths=0.55, medianprops={"color": "black", "linewidth": 1.5},
)
for patch, treatment in zip(boxes["boxes"], ordered):
    patch.set_facecolor(COLOR_MAP.get(treatment, "#AAAAAA"))
    patch.set_alpha(0.8)

rng = np.random.default_rng(42)
for i, group in enumerate(values, start=1):
    ax.scatter(
        i + rng.uniform(-0.1, 0.1, len(group)), group,
        s=32, color="black", alpha=0.65, zorder=3,
    )

all_values = np.concatenate(values)
data_min, data_max = float(all_values.min()), float(all_values.max())
span = max(data_max - data_min, 1.0e-6)
if span < 0.01:
    span = 1.0
for i, (treatment, group) in enumerate(zip(ordered, values), start=1):
    if letters[treatment]:
        ax.text(
            i, float(group.max()) + span * 0.07, letters[treatment],
            ha="center", va="bottom", fontsize=13, fontweight="bold",
        )

ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.7)
ax.set_ylabel("Control-referenced Z-score")
ax.set_xlabel("Treatment")
p_label = f"ANOVA p = {anova_p:.3g}" if np.isfinite(anova_p) else "ANOVA unavailable"
ax.set_title(
    f"{ANTIBODY_PANEL} | {EXPERIMENT_ID} | {PLOT_COMPARTMENT}\n"
    f"{plot_feature} | {p_label}"
)
ax.set_ylim(data_min - span * 0.12, data_max + span * 0.22)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.tight_layout()

# Use the output directory defined in section 6.
plot_stem = f"{safe_name(PLOT_COMPARTMENT)}_{safe_name(plot_feature)}"
fig.savefig(output_dir / f"{plot_stem}.svg", bbox_inches="tight")
fig.savefig(output_dir / f"{plot_stem}.png", dpi=300, bbox_inches="tight")
if not tukey_table.empty:
    tukey_table.to_csv(output_dir / f"{plot_stem}_tukey.csv", index=False)
pd.DataFrame({
    "Treatment": ordered,
    "Letters": [letters[t] for t in ordered],
    "N_wells": [len(v) for v in values],
    "Mean_Zscore": [float(np.mean(v)) for v in values],
    "SD_Zscore": [float(np.std(v, ddof=1)) if len(v) > 1 else np.nan for v in values],
    "ANOVA_F": anova_f,
    "ANOVA_p": anova_p,
}).to_csv(output_dir / f"{plot_stem}_statistics.csv", index=False)
plt.show()
